# Automated VWAP V1 Backtest Runner

This notebook is a clean automated backtest runner for the VWAP Probability Band Engine project.

It tests one defined automated strategy version on historical OHLC CSV data.

The notebook is separate from the research notebook and separate from the visual/discretionary MT5 overlay.

This notebook does not connect to MT5 and does not place live trades.

## Strategy Version

**Version:** Automated VWAP V1

The current V1 candidate uses:

- continuation-only logic
- both longs and shorts allowed
- green reclaim / lower-green rejection structure
- weak directional red-shift filter
- entry at next bar open
- fixed 29-point stop loss
- fixed 58-point take profit
- breakeven after +29 points
- 1% risk per trade
- stop after 2 consecutive daily losses
- stop after +8% realised daily profit
- no new trades after 19:00 Europe/London

The aim is to make this notebook a clean V1 runner that can later be duplicated into V2, V3, V4, or converted into a `.py` backtest/live runner.

## 1. Project Setup

This section imports the required libraries, detects the project root, and defines the main project folders used by the notebook.

In [1]:
from pathlib import Path
from pprint import pprint
import json

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a .git folder is found.

    This allows the notebook to work whether it is run from the project root
    or from inside the notebooks folder.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / ".git").exists():
            return path

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
HISTORICAL_DATA_DIR = DATA_DIR / "historical"
SRC_DIR = PROJECT_ROOT / "src"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "automated_vwap_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Historical data folder:", HISTORICAL_DATA_DIR)
print("Source folder:", SRC_DIR)
print("Output folder:", OUTPUT_DIR)

Current working directory: c:\GitHub Projects\VWAP-probability-band-engine\notebooks
Detected project root: C:\GitHub Projects\VWAP-probability-band-engine
Historical data folder: C:\GitHub Projects\VWAP-probability-band-engine\data\historical
Source folder: C:\GitHub Projects\VWAP-probability-band-engine\src
Output folder: C:\GitHub Projects\VWAP-probability-band-engine\artifacts\automated_vwap_v1


## 2. Editable Strategy Configuration

This is the main control panel for the V1 backtest.

Most basic V1 experiments should be possible by changing this cell only.

In [2]:
CONFIG = {
    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "csv_file": "US100_cash_M1_NY_session_30d.csv",
    "dataset_name": "US100_cash_M1_NY_session_30d",

    # ------------------------------------------------------------------
    # Strategy identity
    # ------------------------------------------------------------------
    "strategy_version": "automated_vwap_v1",
    "strategy_description": "Continuation VWAP reclaim/rejection with weak directional red-shift filter",

    # ------------------------------------------------------------------
    # Session handling
    # ------------------------------------------------------------------
    "session_timezone": "Europe/London",
    "no_new_trades_after": "19:00",

    # ------------------------------------------------------------------
    # Direction controls
    # ------------------------------------------------------------------
    "allow_longs": True,
    "allow_shorts": True,

    # ------------------------------------------------------------------
    # Strategy filters
    # ------------------------------------------------------------------
    "strategy_family": "continuation_only",
    "strategy_filter": "weak_red_shift_only",
    "red_shift_filter_choice": "weak_directional",

    # ------------------------------------------------------------------
    # Entry logic
    # ------------------------------------------------------------------
    "entry_timing": "next_bar_open",

    # ------------------------------------------------------------------
    # Fixed Nasdaq point trade levels
    # ------------------------------------------------------------------
    "sl_points": 29.0,
    "tp_points": 58.0,
    "be_trigger_points": 29.0,

    # ------------------------------------------------------------------
    # Risk controls
    # ------------------------------------------------------------------
    "risk_per_trade_pct": 1.0,
    "daily_max_consecutive_losses": 2,
    "daily_profit_cap_pct": 8.0,

    # ------------------------------------------------------------------
    # Candle quality filters
    # ------------------------------------------------------------------
    "min_body_ratio": 0.30,
    "min_close_through_green": 0.0,
    "max_extension_from_green": 35.0,

    # ------------------------------------------------------------------
    # Output settings
    # ------------------------------------------------------------------
    "save_trade_log": True,
    "save_daily_summary": True,
    "save_skipped_signals": True,
    "save_config_snapshot": True,
}

pprint(CONFIG)

{'allow_longs': True,
 'allow_shorts': True,
 'be_trigger_points': 29.0,
 'csv_file': 'US100_cash_M1_NY_session_30d.csv',
 'daily_max_consecutive_losses': 2,
 'daily_profit_cap_pct': 8.0,
 'dataset_name': 'US100_cash_M1_NY_session_30d',
 'entry_timing': 'next_bar_open',
 'max_extension_from_green': 35.0,
 'min_body_ratio': 0.3,
 'min_close_through_green': 0.0,
 'no_new_trades_after': '19:00',
 'red_shift_filter_choice': 'weak_directional',
 'risk_per_trade_pct': 1.0,
 'save_config_snapshot': True,
 'save_daily_summary': True,
 'save_skipped_signals': True,
 'save_trade_log': True,
 'session_timezone': 'Europe/London',
 'sl_points': 29.0,
 'strategy_description': 'Continuation VWAP reclaim/rejection with weak '
                         'directional red-shift filter',
 'strategy_family': 'continuation_only',
 'strategy_filter': 'weak_red_shift_only',
 'strategy_version': 'automated_vwap_v1',
 'tp_points': 58.0}


## 3. Trade Level Sanity Check

This section checks that the configured stop loss, take profit, and breakeven levels behave correctly for both long and short trades.

The config stores all point distances as positive numbers.

The trade simulator will later convert those distances into the correct long or short price levels.

In [3]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels for a long or short trade.
    """
    side = side.lower()

    sl = float(config["sl_points"])
    tp = float(config["tp_points"])
    be = float(config["be_trigger_points"])

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be either 'long' or 'short'")


example_entry = 20000.0

level_check = {
    "long_example": build_trade_levels(example_entry, "long", CONFIG),
    "short_example": build_trade_levels(example_entry, "short", CONFIG),
}

pprint(level_check)

{'long_example': {'breakeven_trigger_price': 20029.0,
                  'entry_price': 20000.0,
                  'side': 'long',
                  'stop_price': 19971.0,
                  'target_price': 20058.0},
 'short_example': {'breakeven_trigger_price': 19971.0,
                   'entry_price': 20000.0,
                   'side': 'short',
                   'stop_price': 20029.0,
                   'target_price': 19942.0}}


## 4. Data Loading

This section loads the configured OHLC CSV file.

It handles common MT5-style column names and standardises them into:

- `datetime`
- `open`
- `high`
- `low`
- `close`

Optional fields such as `tick_volume`, `spread`, and `real_volume` are preserved when available.

In [4]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

COLUMN_ALIASES = {
    "datetime": [
        "datetime",
        "time",
        "timestamp",
        "date",
        "Date",
        "Time",
        "Datetime",
        "Local time",
        "Gmt time",
        "GMT time",
    ],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],
    "tick_volume": [
        "tick_volume",
        "volume",
        "Volume",
        "tickvol",
        "Tick Volume",
        "Tick volume",
    ],
    "spread": ["spread", "Spread"],
    "real_volume": ["real_volume", "Real Volume", "real volume"],
}


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of possible column names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    lower_map = {str(col).lower(): col for col in df.columns}

    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def standardise_raw_ohlc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common OHLC column names into the standard names used by this notebook.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """
    Validate that the dataframe contains the required columns.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )


def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw OHLC data for the automated V1 backtest.

    This function only cleans and validates raw market data.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing VWAP engine logic.
    """
    out = standardise_raw_ohlc_columns(df)

    validate_required_columns(out, REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).copy()

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).copy()

    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    out = out.sort_values("datetime").reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out


def list_candidate_data_files() -> list[Path]:
    """
    Find likely CSV or Parquet files inside the project.
    """
    patterns = ["*.csv", "*.parquet"]
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    files = []

    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


def resolve_data_file(config: dict) -> Path:
    """
    Resolve the configured data file.

    The config can use:
    - a file name, e.g. US100_cash_M1_NY_session_30d.csv
    - a relative path, e.g. data/historical/file.csv
    - an absolute path
    """
    configured_file = Path(config["csv_file"])

    if configured_file.is_absolute() and configured_file.exists():
        return configured_file

    direct_project_path = PROJECT_ROOT / configured_file

    if direct_project_path.exists():
        return direct_project_path

    candidate_files = list_candidate_data_files()

    matching_files = [
        file for file in candidate_files
        if file.name == configured_file.name
    ]

    if len(matching_files) == 1:
        return matching_files[0]

    if len(matching_files) > 1:
        print("Multiple matching files found:")

        for file in matching_files:
            print("-", file.relative_to(PROJECT_ROOT))

        raise ValueError(
            f"Multiple files named {configured_file.name} were found. "
            "Use a more specific relative path in CONFIG['csv_file']."
        )

    print(f"Could not find configured file: {config['csv_file']}")
    print("\nAvailable candidate files:")

    for file in candidate_files[:100]:
        print("-", file.relative_to(PROJECT_ROOT))

    raise FileNotFoundError(
        f"Could not find configured data file: {config['csv_file']}"
    )


def load_market_data(config: dict) -> tuple[pd.DataFrame, Path]:
    """
    Load the configured CSV or Parquet file and return a cleaned OHLC dataframe.
    """
    data_file = resolve_data_file(config)

    if data_file.suffix.lower() == ".csv":
        raw_df = pd.read_csv(data_file)
    elif data_file.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(data_file)
    else:
        raise ValueError(f"Unsupported file type: {data_file.suffix}")

    prepared_df = prepare_raw_ohlc_dataframe(raw_df)

    return prepared_df, data_file

In [5]:
raw_ohlc_df, DATA_FILE = load_market_data(CONFIG)

print(f"Loaded data file: {DATA_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows loaded: {len(raw_ohlc_df):,}")
print(f"Start datetime: {raw_ohlc_df['datetime'].min()}")
print(f"End datetime: {raw_ohlc_df['datetime'].max()}")

print("\nColumns:")
print(list(raw_ohlc_df.columns))

print("\nFirst 5 rows:")
print(raw_ohlc_df.head().to_string(index=False))

Loaded data file: data\historical\US100_cash_M1_NY_session_30d.csv
Rows loaded: 6,728
Start datetime: 2026-01-29 15:22:00
End datetime: 2026-03-17 16:29:00

Columns:
['datetime', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'candle_range', 'candle_body', 'body_ratio']

First 5 rows:
           datetime     open     high      low    close  tick_volume  spread  real_volume  candle_range  candle_body  body_ratio
2026-01-29 15:22:00 26033.45 26035.15 26027.95 26034.55          261     190            0           7.2          1.1    0.152778
2026-01-29 15:23:00 26034.45 26037.55 26032.15 26034.25          234     190            0           5.4          0.2    0.037037
2026-01-29 15:24:00 26034.45 26040.05 26034.45 26040.05          234     190            0           5.6          5.6    1.000000
2026-01-29 15:25:00 26040.15 26044.45 26036.25 26044.25          277     190            0           8.2          4.1    0.500000
2026-01-29 15:26:00 26044.55 26046.65 26039.

## 5. Dataset Summary

This section prints a compact summary of the loaded dataset before the VWAP engine is applied.

In [6]:
def summarise_raw_ohlc_data(df: pd.DataFrame, config: dict, data_file: Path) -> dict:
    """
    Create a compact summary of the loaded OHLC dataset.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "configured_file": config["csv_file"],
        "resolved_file": str(data_file.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "open_min": df["open"].min(),
        "open_max": df["open"].max(),
        "high_max": df["high"].max(),
        "low_min": df["low"].min(),
        "close_min": df["close"].min(),
        "close_max": df["close"].max(),
        "mean_candle_range": df["candle_range"].mean(),
        "median_candle_range": df["candle_range"].median(),
        "mean_body_ratio": df["body_ratio"].mean(),
        "zero_range_candles": int((df["candle_range"] <= 0).sum()),
        "duplicate_datetimes": int(df["datetime"].duplicated().sum()),
    }

    return summary


dataset_summary = summarise_raw_ohlc_data(raw_ohlc_df, CONFIG, DATA_FILE)

pprint(dataset_summary)

{'close_max': np.float64(26074.05),
 'close_min': np.float64(24321.73),
 'configured_file': 'US100_cash_M1_NY_session_30d.csv',
 'dataset_name': 'US100_cash_M1_NY_session_30d',
 'duplicate_datetimes': 0,
 'end_datetime': Timestamp('2026-03-17 16:29:00'),
 'high_max': np.float64(26076.75),
 'low_min': np.float64(24314.73),
 'mean_body_ratio': np.float64(0.4762610225933656),
 'mean_candle_range': np.float64(16.606730083234257),
 'median_candle_range': np.float64(11.900000000001455),
 'open_max': np.float64(26074.15),
 'open_min': np.float64(24320.03),
 'resolved_file': 'data\\historical\\US100_cash_M1_NY_session_30d.csv',
 'rows': 6728,
 'start_datetime': Timestamp('2026-01-29 15:22:00'),
 'zero_range_candles': 0}


## 6. Ready for VWAP Engine Integration

At this point, the notebook has:

- loaded the configured dataset
- standardised OHLC column names
- parsed timestamps
- sorted candles chronologically
- created basic candle features
- printed a dataset summary

The next section will pass `raw_ohlc_df` into the existing VWAP Probability Band Engine from `src/`.

In [7]:
print("raw_ohlc_df is ready for VWAP engine integration.")
print(f"Shape: {raw_ohlc_df.shape}")

print("\nLast 5 rows:")
print(raw_ohlc_df.tail().to_string(index=False))

raw_ohlc_df is ready for VWAP engine integration.
Shape: (6728, 11)

Last 5 rows:
           datetime     open     high      low    close  tick_volume  spread  real_volume  candle_range  candle_body  body_ratio
2026-03-17 16:25:00 24785.45 24794.45 24781.55 24792.35          398     190            0          12.9          6.9    0.534884
2026-03-17 16:26:00 24792.25 24792.55 24779.95 24782.25          342     190            0          12.6         10.0    0.793651
2026-03-17 16:27:00 24781.95 24783.35 24771.25 24772.75          370     190            0          12.1          9.2    0.760331
2026-03-17 16:28:00 24772.55 24775.05 24767.55 24775.05          420     190            0           7.5          2.5    0.333333
2026-03-17 16:29:00 24774.85 24782.25 24767.05 24769.75          405     190            0          15.2          5.1    0.335526


## 7. Existing VWAP Engine Integration

This section passes the cleaned OHLC data through the existing VWAP Probability Band Engine from `src/`.

The aim is to reuse the existing project logic instead of rewriting or changing the model.

This section creates:

- VWAP/reference line
- sigma estimate
- upper and lower probability bands
- z-score
- zone classification
- automation-friendly band aliases

In [8]:
import sys
import importlib


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import src.config as engine_config_module
importlib.reload(engine_config_module)

from src.config import CONFIG as ENGINE_CONFIG
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import compute_zscore, classify_zones_series


print("Loaded existing VWAP engine modules.")
print("Engine config snapshot:")
print(f"- reference_type: {ENGINE_CONFIG.get('reference_type')}")
print(f"- vol_method: {ENGINE_CONFIG.get('vol_method')}")
print(f"- zone_thresholds: {ENGINE_CONFIG.get('zone_thresholds')}")

Loaded existing VWAP engine modules.
Engine config snapshot:
- reference_type: VWAP
- vol_method: ewma
- zone_thresholds: [0.5, 1.0, 2.0]


## 8. Engine Output Builder

This section wraps the existing VWAP engine into one clean function.

The function receives cleaned OHLC data and returns a full engine dataframe with standardised automation columns.

The strategy config remains separate from the engine config.

In [9]:
REQUIRED_ENGINE_OUTPUT_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
]


ENGINE_COLUMN_ALIASES = {
    "vwap": ["vwap", "VWAP", "reference", "ref", "reference_line"],
    "upper_green": ["upper_green", "upper_1", "upper_band_1", "band_1p", "band_1_plus", "band_1+", "z1_upper", "upper_sigma_1"],
    "upper_orange": ["upper_orange", "upper_2", "upper_band_2", "band_2p", "band_2_plus", "band_2+", "z2_upper", "upper_sigma_2"],
    "upper_red": ["upper_red", "upper_3", "upper_band_3", "band_3p", "band_3_plus", "band_3+", "z3_upper", "upper_sigma_3"],
    "lower_green": ["lower_green", "lower_1", "lower_band_1", "band_1n", "band_1m", "band_1_minus", "band_1-", "z1_lower", "lower_sigma_1"],
    "lower_orange": ["lower_orange", "lower_2", "lower_band_2", "band_2n", "band_2m", "band_2_minus", "band_2-", "z2_lower", "lower_sigma_2"],
    "lower_red": ["lower_red", "lower_3", "lower_band_3", "band_3n", "band_3m", "band_3_minus", "band_3-", "z3_lower", "lower_sigma_3"],
}


def add_engine_band_aliases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add automation-friendly VWAP/band aliases to the existing engine output.

    This does not change model values.
    It only creates easier column names for the automated strategy layer.
    """
    out = df.copy()

    for standard_name, aliases in ENGINE_COLUMN_ALIASES.items():
        if standard_name in out.columns:
            continue

        matched_col = find_column(out, aliases)

        if matched_col is not None:
            out[standard_name] = out[matched_col]

    return out


def validate_engine_output_columns(df: pd.DataFrame) -> None:
    """
    Confirm that the engine output contains the columns needed by the automated strategy layer.
    """
    missing = [col for col in REQUIRED_ENGINE_OUTPUT_COLUMNS if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required engine output columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )

    print("All required engine output columns are available.")


def build_existing_engine_output(raw_ohlc_df: pd.DataFrame, engine_config: dict) -> pd.DataFrame:
    """
    Build VWAP probability-band output from raw OHLC data using the existing project logic.

    This function does not modify the existing model.

    It calls the existing source functions and then adds automation-friendly aliases.
    """
    df = raw_ohlc_df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    if "tick_volume" not in df.columns:
        df["tick_volume"] = 1.0

    df["tick_volume"] = (
        pd.to_numeric(df["tick_volume"], errors="coerce")
        .fillna(1.0)
        .clip(lower=1.0)
    )

    df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    df["session_date"] = df["datetime"].dt.date

    df["reference"] = compute_reference(df, engine_config)
    df["price_deviation"] = df["close"] - df["reference"]

    df["sigma"] = compute_sigma(df, engine_config)

    bands = compute_bands(df, df["sigma"])
    df = pd.concat([df, bands], axis=1)

    df["z_score"] = compute_zscore(df)
    df["zone"] = classify_zones_series(df["z_score"], engine_config["zone_thresholds"])

    df = add_engine_band_aliases(df)

    df["candle_range"] = df["high"] - df["low"]
    df["candle_body"] = (df["close"] - df["open"]).abs()
    df["body_ratio"] = np.where(
        df["candle_range"] > 0,
        df["candle_body"] / df["candle_range"],
        0.0,
    )

    validate_engine_output_columns(df)

    return df

## 9. Run Existing VWAP Engine

This section runs the existing engine on the loaded OHLC data.

The output is the full engine dataframe used by the automated V1 strategy.

In [10]:
engine_df = build_existing_engine_output(raw_ohlc_df, ENGINE_CONFIG)

print(f"Built engine output for {len(engine_df):,} rows.")

engine_preview_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "sigma",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

print("\nEngine output preview:")
print(engine_df[engine_preview_cols].tail(10).to_string(index=False))

✅ VWAP computed — resets per session: True
✅ Sigma computed (ewma)
   Mean sigma : 37.132500
   Min sigma  : 14.478627
   Max sigma  : 145.465266
   Floor used : 14.478627
All required engine output columns are available.
Built engine output for 6,728 rows.

Engine output preview:
                 datetime     open     high      low    close         vwap     sigma  upper_green  upper_orange    upper_red  lower_green  lower_orange    lower_red  z_score zone
2026-03-17 16:20:00+00:00 24767.45 24778.35 24767.35 24775.05 24637.921835 25.951377 24663.873212  24689.824588 24715.775965 24611.970459  24586.019082 24560.067706 5.284042  Z3+
2026-03-17 16:21:00+00:00 24774.45 24774.55 24766.35 24769.05 24638.898588 26.594278 24665.492866  24692.087145 24718.681423 24612.304310  24585.710031 24559.115753 4.893963  Z3+
2026-03-17 16:22:00+00:00 24768.55 24768.95 24764.85 24765.55 24639.828921 26.841566 24666.670487  24693.512053 24720.353618 24612.987355  24586.145789 24559.304223 4.683821  Z3+
20

## 10. Automation-Ready DataFrame

This section prepares the engine output for the automated execution layer.

The output dataframe should contain all OHLC, VWAP, and band columns needed for feature engineering and signal generation.

In [11]:
def prepare_automation_dataframe(engine_df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare existing VWAP engine output for the automated strategy layer.

    This function validates and cleans the columns required for automated V1 signal generation.
    """
    out = engine_df.copy()

    out = add_engine_band_aliases(out)
    validate_engine_output_columns(out)

    out["datetime"] = pd.to_datetime(out["datetime"], utc=True, errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    optional_numeric_cols = [
        "sigma",
        "z_score",
        "tick_volume",
        "spread",
        "real_volume",
    ]

    for col in numeric_cols + optional_numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    out["bar_index"] = np.arange(len(out))

    return out


automation_ready_df = prepare_automation_dataframe(engine_df)

print(f"Automation-ready dataframe rows: {len(automation_ready_df):,}")
print(f"Automation-ready dataframe columns: {len(automation_ready_df.columns):,}")

automation_preview_cols = [
    "bar_index",
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "body_ratio",
]

print("\nAutomation-ready preview:")
print(automation_ready_df[automation_preview_cols].tail(10).to_string(index=False))

All required engine output columns are available.
Automation-ready dataframe rows: 6,728
Automation-ready dataframe columns: 32

Automation-ready preview:
 bar_index                  datetime    close         vwap  upper_green  upper_orange    upper_red  lower_green  lower_orange    lower_red  body_ratio
      6718 2026-03-17 16:20:00+00:00 24775.05 24637.921835 24663.873212  24689.824588 24715.775965 24611.970459  24586.019082 24560.067706    0.690909
      6719 2026-03-17 16:21:00+00:00 24769.05 24638.898588 24665.492866  24692.087145 24718.681423 24612.304310  24585.710031 24559.115753    0.658537
      6720 2026-03-17 16:22:00+00:00 24765.55 24639.828921 24666.670487  24693.512053 24720.353618 24612.987355  24586.145789 24559.304223    0.731707
      6721 2026-03-17 16:23:00+00:00 24772.25 24640.869985 24668.178788  24695.487591 24722.796393 24613.561182  24586.252379 24558.943576    0.708333
      6722 2026-03-17 16:24:00+00:00 24785.85 24642.219173 24670.659071  24699.098968 2472

## 11. Engine Summary

This section prints a compact summary of the VWAP engine output.

It helps confirm that the probability bands were generated correctly before signal rules are applied.

In [12]:
def summarise_engine_output(df: pd.DataFrame, config: dict, engine_config: dict) -> dict:
    """
    Create a compact summary of the engine output.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "strategy_version": config["strategy_version"],
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "reference_type": engine_config.get("reference_type"),
        "vol_method": engine_config.get("vol_method"),
        "zone_thresholds": engine_config.get("zone_thresholds"),
        "mean_sigma": df["sigma"].mean() if "sigma" in df.columns else None,
        "median_sigma": df["sigma"].median() if "sigma" in df.columns else None,
        "mean_green_band_width": (df["upper_green"] - df["lower_green"]).mean(),
        "median_green_band_width": (df["upper_green"] - df["lower_green"]).median(),
        "mean_red_band_width": (df["upper_red"] - df["lower_red"]).mean(),
        "median_red_band_width": (df["upper_red"] - df["lower_red"]).median(),
        "zone_counts": df["zone"].value_counts(dropna=False).to_dict() if "zone" in df.columns else None,
        "missing_vwap_values": int(df["vwap"].isna().sum()),
        "missing_upper_green_values": int(df["upper_green"].isna().sum()),
        "missing_lower_green_values": int(df["lower_green"].isna().sum()),
    }

    return summary


engine_summary = summarise_engine_output(automation_ready_df, CONFIG, ENGINE_CONFIG)

pprint(engine_summary)

{'dataset_name': 'US100_cash_M1_NY_session_30d',
 'end_datetime': Timestamp('2026-03-17 16:29:00+0000', tz='UTC'),
 'mean_green_band_width': np.float64(74.26500004861145),
 'mean_red_band_width': np.float64(222.7950001458344),
 'mean_sigma': np.float64(37.13250002430573),
 'median_green_band_width': np.float64(58.681587074392155),
 'median_red_band_width': np.float64(176.04476122317647),
 'median_sigma': np.float64(29.34079353719643),
 'missing_lower_green_values': 0,
 'missing_upper_green_values': 0,
 'missing_vwap_values': 0,
 'reference_type': 'VWAP',
 'rows': 6728,
 'start_datetime': Timestamp('2026-01-29 15:22:00+0000', tz='UTC'),
 'strategy_version': 'automated_vwap_v1',
 'vol_method': 'ewma',
 'zone_counts': {'Z0': 2212,
                 'Z1+': 484,
                 'Z1-': 652,
                 'Z2+': 923,
                 'Z2-': 841,
                 'Z3+': 747,
                 'Z3-': 869},
 'zone_thresholds': [0.5, 1.0, 2.0]}


## 12. Ready for Automation Feature Engineering

At this point, the notebook has:

- loaded raw OHLC data
- reused the existing VWAP engine from `src/`
- created VWAP and probability bands
- added automation-friendly band aliases
- validated the automation-ready dataframe
- printed an engine summary

The next section will create V1 automation features from `automation_ready_df`.

In [13]:
print("automation_ready_df is ready for V1 feature engineering.")
print(f"Shape: {automation_ready_df.shape}")

print("\nAvailable key columns:")
key_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

for col in key_cols:
    print(f"- {col}: {'yes' if col in automation_ready_df.columns else 'missing'}")

automation_ready_df is ready for V1 feature engineering.
Shape: (6728, 32)

Available key columns:
- datetime: yes
- open: yes
- high: yes
- low: yes
- close: yes
- vwap: yes
- upper_green: yes
- upper_orange: yes
- upper_red: yes
- lower_green: yes
- lower_orange: yes
- lower_red: yes
- z_score: yes
- zone: yes
